In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


In [4]:
# ============================================================
# CN7 - 논문 기반 전처리 + 파생변수 + 불균형 처리 + 모델 비교
# Target : PassOrFail
#
# 실험
# 0. Baseline
# 1. 상수 제거
# 2. 공정 기반 파생변수
# 3. Class Weight
# 4. SMOTE
# 5. ADASYN
# 6. SMOTE + Tomek
#
# 모델
# - Logistic Regression
# - Random Forest
# - XGBoost
#
# 평가
# - Accuracy
# - Precision
# - Recall
# - F1
# - PR-AUC
# ============================================================


# ============================================================
# 0. 라이브러리
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 기본 설정
# ============================================================

TARGET = "PassOrFail"
RANDOM_STATE = 42

df_cn7 = train_cn7.copy()

print("=" * 70)
print("CN7 데이터")
print("=" * 70)

print("Shape :", df_cn7.shape)
print("\nTarget 분포")
print(df_cn7[TARGET].value_counts())

print("\nTarget 비율")
print(df_cn7[TARGET].value_counts(normalize=True))


# ============================================================
# 2. 결측치 확인
# ============================================================

missing = pd.DataFrame({
    "count": df_cn7.isnull().sum(),
    "ratio": df_cn7.isnull().mean()
})

missing = missing[missing["count"] > 0]

print("\n" + "=" * 70)
print("결측치")
print("=" * 70)

if len(missing) == 0:
    print("결측치 없음")

else:
    display(missing.sort_values("ratio", ascending=False))


# ============================================================
# 3. 완전 중복 확인
# ============================================================

print("\n" + "=" * 70)
print("완전 중복")
print("=" * 70)

print("완전 중복 행 :", df_cn7.duplicated().sum())


# ============================================================
# 4. Target 제외 동일 공정조건 중복 확인
# ============================================================

feature_cols = [
    c for c in df_cn7.columns
    if c != TARGET
]

same_x = df_cn7.duplicated(
    subset=feature_cols,
    keep=False
)

print("\n동일 공정조건 데이터 :", same_x.sum())


# 동일 X에서 Target이 다른 경우
conflict = (
    df_cn7
    .groupby(feature_cols, dropna=False)[TARGET]
    .nunique()
)

conflict = conflict[conflict > 1]

print("동일 X + 다른 Target 조건 수 :", len(conflict))


# ============================================================
# 5. 상수 컬럼 제거
# ============================================================

constant_cols = [
    c for c in feature_cols
    if df_cn7[c].nunique(dropna=False) <= 1
]

print("\n" + "=" * 70)
print("상수 컬럼")
print("=" * 70)

print(constant_cols)
print("개수 :", len(constant_cols))

df_cn7 = df_cn7.drop(
    columns=constant_cols
)

print("상수 제거 후 Shape :", df_cn7.shape)


# ============================================================
# 6. 결측치 처리
#
# 이미 결측치가 없다면 아무 변화 없음
# 극소수 결측값은 median 사용
# ============================================================

numeric_cols = (
    df_cn7
    .drop(columns=[TARGET])
    .select_dtypes(include=np.number)
    .columns
)

for col in numeric_cols:

    if df_cn7[col].isnull().sum() > 0:

        df_cn7[col] = (
            df_cn7[col]
            .fillna(df_cn7[col].median())
        )


print("\n결측치 처리 후 :", df_cn7.isnull().sum().sum())


# ============================================================
# 7. 공정 기반 파생변수 생성 함수
#
# 주의:
# 현재 데이터가 Standard Scaling 되어 있으므로
# 물리적 단위 자체를 의미하는 비율 변수는 최소화.
#
# 차이 / 편차 / 범위 중심으로 생성
# ============================================================

def add_cn7_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # 1) 사출시간 - 충전시간 차이
    # --------------------------------------------------------

    if {
        "Injection_Time",
        "Filling_Time"
    }.issubset(df.columns):

        df["Injection_Filling_Diff"] = (
            df["Injection_Time"]
            - df["Filling_Time"]
        )


    # --------------------------------------------------------
    # 2) 가소화시간 - 충전시간 차이
    # --------------------------------------------------------

    if {
        "Plasticizing_Time",
        "Filling_Time"
    }.issubset(df.columns):

        df["Plasticizing_Filling_Diff"] = (
            df["Plasticizing_Time"]
            - df["Filling_Time"]
        )


    # --------------------------------------------------------
    # 3) Cycle Time과 Injection Time 차이
    # --------------------------------------------------------

    if {
        "Cycle_Time",
        "Injection_Time"
    }.issubset(df.columns):

        df["Cycle_Injection_Diff"] = (
            df["Cycle_Time"]
            - df["Injection_Time"]
        )


    # --------------------------------------------------------
    # 4) Cycle Time과 Plasticizing Time 차이
    # --------------------------------------------------------

    if {
        "Cycle_Time",
        "Plasticizing_Time"
    }.issubset(df.columns):

        df["Cycle_Plasticizing_Diff"] = (
            df["Cycle_Time"]
            - df["Plasticizing_Time"]
        )


    # --------------------------------------------------------
    # 5) 최대 Screw RPM - 평균 Screw RPM
    # --------------------------------------------------------

    if {
        "Max_Screw_RPM",
        "Average_Screw_RPM"
    }.issubset(df.columns):

        df["Screw_RPM_Gap"] = (
            df["Max_Screw_RPM"]
            - df["Average_Screw_RPM"]
        )


    # --------------------------------------------------------
    # 6) Injection Pressure - Switch Over Pressure
    # --------------------------------------------------------

    if {
        "Max_Injection_Pressure",
        "Max_Switch_Over_Pressure"
    }.issubset(df.columns):

        df["Injection_Switch_Pressure_Gap"] = (
            df["Max_Injection_Pressure"]
            - df["Max_Switch_Over_Pressure"]
        )


    # --------------------------------------------------------
    # 7) Max Back Pressure - Average Back Pressure
    # --------------------------------------------------------

    if {
        "Max_Back_Pressure",
        "Average_Back_Pressure"
    }.issubset(df.columns):

        df["Back_Pressure_Gap"] = (
            df["Max_Back_Pressure"]
            - df["Average_Back_Pressure"]
        )


    # --------------------------------------------------------
    # 8) Plasticizing Position - Cushion Position
    # --------------------------------------------------------

    if {
        "Plasticizing_Position",
        "Cushion_Position"
    }.issubset(df.columns):

        df["Plasticizing_Cushion_Gap"] = (
            df["Plasticizing_Position"]
            - df["Cushion_Position"]
        )


    # --------------------------------------------------------
    # 9) Clamp Open Position - Plasticizing Position
    # --------------------------------------------------------

    if {
        "Clamp_Open_Position",
        "Plasticizing_Position"
    }.issubset(df.columns):

        df["Clamp_Plasticizing_Position_Gap"] = (
            df["Clamp_Open_Position"]
            - df["Plasticizing_Position"]
        )


    # --------------------------------------------------------
    # 10) Barrel Temperature 계열
    # --------------------------------------------------------

    barrel_cols = [
        c for c in df.columns
        if c.startswith("Barrel_Temperature")
    ]

    if len(barrel_cols) >= 2:

        df["Barrel_Temp_Mean"] = (
            df[barrel_cols].mean(axis=1)
        )

        df["Barrel_Temp_Std"] = (
            df[barrel_cols].std(axis=1)
        )

        df["Barrel_Temp_Range"] = (
            df[barrel_cols].max(axis=1)
            - df[barrel_cols].min(axis=1)
        )


    # --------------------------------------------------------
    # 11) Mold Temperature 계열이 존재할 경우
    # --------------------------------------------------------

    mold_temp_cols = [
        c for c in df.columns
        if "Mold_Temperature" in c
    ]

    if len(mold_temp_cols) >= 2:

        df["Mold_Temp_Mean"] = (
            df[mold_temp_cols].mean(axis=1)
        )

        df["Mold_Temp_Std"] = (
            df[mold_temp_cols].std(axis=1)
        )

        df["Mold_Temp_Range"] = (
            df[mold_temp_cols].max(axis=1)
            - df[mold_temp_cols].min(axis=1)
        )


    return df


# ============================================================
# 8. 파생변수 생성
# ============================================================

original_cols = set(df_cn7.columns)

df_cn7_fe = add_cn7_features(
    df_cn7
)

new_cols = [
    c for c in df_cn7_fe.columns
    if c not in original_cols
]

print("\n" + "=" * 70)
print("생성된 파생변수")
print("=" * 70)

for col in new_cols:
    print(col)

print("\n파생변수 개수 :", len(new_cols))
print("최종 Shape :", df_cn7_fe.shape)


# ============================================================
# 9. inf / NaN 최종 확인
# ============================================================

df_cn7_fe = df_cn7_fe.replace(
    [np.inf, -np.inf],
    np.nan
)

feature_cols = [
    c for c in df_cn7_fe.columns
    if c != TARGET
]

for col in feature_cols:

    if df_cn7_fe[col].isnull().sum() > 0:

        df_cn7_fe[col] = (
            df_cn7_fe[col]
            .fillna(df_cn7_fe[col].median())
        )


print("\n최종 결측치 :", df_cn7_fe.isnull().sum().sum())


# ============================================================
# 10. X / y 분리
# ============================================================

X = df_cn7_fe.drop(
    columns=[TARGET]
)

y = df_cn7_fe[TARGET].astype(int)

print("\nX Shape :", X.shape)
print("y Shape :", y.shape)


# ============================================================
# 11. Train / Test Split
#
# 매우 중요:
# 오버샘플링 전에 split
# Test에는 SMOTE/ADASYN 절대 적용 X
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\n" + "=" * 70)
print("Train / Test")
print("=" * 70)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

print("\nTrain Target")
print(y_train.value_counts())

print("\nTest Target")
print(y_test.value_counts())


# ============================================================
# 12. 평가 함수
# ============================================================

results = []


def evaluate_model(
    experiment,
    model_name,
    model,
    X_test,
    y_test
):

    pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):

        proba = model.predict_proba(X_test)[:, 1]

    else:

        proba = pred


    accuracy = accuracy_score(
        y_test,
        pred
    )

    precision = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    pr_auc = average_precision_score(
        y_test,
        proba
    )


    print("\n")
    print("=" * 70)
    print(f"[{experiment}] {model_name}")
    print("=" * 70)

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1       :", f1)
    print("PR-AUC   :", pr_auc)

    print("\nConfusion Matrix")
    print(
        confusion_matrix(
            y_test,
            pred
        )
    )

    print("\nClassification Report")

    print(
        classification_report(
            y_test,
            pred,
            digits=4,
            zero_division=0
        )
    )


    results.append({

        "Experiment": experiment,
        "Model": model_name,

        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "PR_AUC": pr_auc

    })


# ============================================================
# 13. 모델 학습 함수
# ============================================================

def run_models(
    experiment,
    X_train,
    y_train,
    X_test,
    y_test,
    class_weight=None
):

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------

    logistic = LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE,
        class_weight=class_weight
    )

    logistic.fit(
        X_train,
        y_train
    )

    evaluate_model(
        experiment,
        "Logistic Regression",
        logistic,
        X_test,
        y_test
    )


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=class_weight
    )

    rf.fit(
        X_train,
        y_train
    )

    evaluate_model(
        experiment,
        "Random Forest",
        rf,
        X_test,
        y_test
    )


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    negative = (y_train == 0).sum()
    positive = (y_train == 1).sum()

    scale_pos_weight = (
        negative / positive
        if positive > 0
        else 1
    )

    # class_weight 실험일 때만
    # XGBoost에서는 scale_pos_weight 사용
    if class_weight == "balanced":

        spw = scale_pos_weight

    else:

        spw = 1


    xgb = XGBClassifier(

        n_estimators=300,

        max_depth=4,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        objective="binary:logistic",

        eval_metric="logloss",

        scale_pos_weight=spw,

        random_state=RANDOM_STATE,

        n_jobs=-1
    )

    xgb.fit(
        X_train,
        y_train
    )

    evaluate_model(
        experiment,
        "XGBoost",
        xgb,
        X_test,
        y_test
    )


# ============================================================
# 14. Experiment 1
# 파생변수 + 불균형 처리 없음
# ============================================================

run_models(
    "No Sampling",
    X_train,
    y_train,
    X_test,
    y_test
)


# ============================================================
# 15. Experiment 2
# Class Weight
# ============================================================

run_models(
    "Class Weight",
    X_train,
    y_train,
    X_test,
    y_test,
    class_weight="balanced"
)


# ============================================================
# 16. SMOTE
#
# 불량 데이터가 극단적으로 적기 때문에
# k_neighbors를 자동 조정
# ============================================================

minority_count = (
    y_train
    .value_counts()
    .min()
)

smote_k = max(
    1,
    min(
        5,
        minority_count - 1
    )
)

print("\nSMOTE k_neighbors :", smote_k)


smote = SMOTE(
    random_state=RANDOM_STATE,
    k_neighbors=smote_k
)

X_train_smote, y_train_smote = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

print("\nSMOTE 전")
print(y_train.value_counts())

print("\nSMOTE 후")
print(y_train_smote.value_counts())


run_models(
    "SMOTE",
    X_train_smote,
    y_train_smote,
    X_test,
    y_test
)


# ============================================================
# 17. ADASYN
# ============================================================

try:

    adasyn_k = max(
        1,
        min(
            5,
            minority_count - 1
        )
    )

    adasyn = ADASYN(
        random_state=RANDOM_STATE,
        n_neighbors=adasyn_k
    )

    X_train_ada, y_train_ada = (
        adasyn.fit_resample(
            X_train,
            y_train
        )
    )

    print("\nADASYN 전")
    print(y_train.value_counts())

    print("\nADASYN 후")
    print(y_train_ada.value_counts())


    run_models(
        "ADASYN",
        X_train_ada,
        y_train_ada,
        X_test,
        y_test
    )


except ValueError as e:

    print("\nADASYN 적용 실패")
    print(e)


# ============================================================
# 18. SMOTE + Tomek
# ============================================================

smote_tomek = SMOTETomek(
    random_state=RANDOM_STATE,
    smote=SMOTE(
        random_state=RANDOM_STATE,
        k_neighbors=smote_k
    )
)

X_train_st, y_train_st = (
    smote_tomek.fit_resample(
        X_train,
        y_train
    )
)

print("\nSMOTE + Tomek 전")
print(y_train.value_counts())

print("\nSMOTE + Tomek 후")
print(y_train_st.value_counts())


run_models(
    "SMOTE + Tomek",
    X_train_st,
    y_train_st,
    X_test,
    y_test
)


# ============================================================
# 19. 전체 결과
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="PR_AUC",
    ascending=False
).reset_index(drop=True)


print("\n")
print("=" * 70)
print("CN7 전체 실험 결과")
print("=" * 70)

display(
    results_df.style.format({

        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "PR_AUC": "{:.4f}"

    })
)


# ============================================================
# 20. F1 기준 결과
# ============================================================

print("\nF1 기준")

display(
    results_df
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 21. Recall 기준 결과
# ============================================================

print("\nRecall 기준")

display(
    results_df
    .sort_values(
        "Recall",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 22. PR-AUC 기준 Best
# ============================================================

best = (
    results_df
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .iloc[0]
)

print("\n")
print("=" * 70)
print("PR-AUC 기준 Best")
print("=" * 70)

print("Experiment :", best["Experiment"])
print("Model      :", best["Model"])
print("Precision  :", round(best["Precision"], 4))
print("Recall     :", round(best["Recall"], 4))
print("F1         :", round(best["F1"], 4))
print("PR-AUC     :", round(best["PR_AUC"], 4))

CN7 데이터
Shape : (1211, 26)

Target 분포
PassOrFail
0    1194
1      17
Name: count, dtype: int64

Target 비율
PassOrFail
0    0.985962
1    0.014038
Name: proportion, dtype: float64

결측치
결측치 없음

완전 중복
완전 중복 행 : 0

동일 공정조건 데이터 : 0
동일 X + 다른 Target 조건 수 : 0

상수 컬럼
['Clamp_Open_Position']
개수 : 1
상수 제거 후 Shape : (1211, 25)

결측치 처리 후 : 0

생성된 파생변수
Injection_Filling_Diff
Plasticizing_Filling_Diff
Cycle_Injection_Diff
Cycle_Plasticizing_Diff
Screw_RPM_Gap
Injection_Switch_Pressure_Gap
Back_Pressure_Gap
Plasticizing_Cushion_Gap
Barrel_Temp_Mean
Barrel_Temp_Std
Barrel_Temp_Range
Mold_Temp_Mean
Mold_Temp_Std
Mold_Temp_Range

파생변수 개수 : 14
최종 Shape : (1211, 39)

최종 결측치 : 0

X Shape : (1211, 38)
y Shape : (1211,)

Train / Test
Train : (968, 38)
Test  : (243, 38)

Train Target
PassOrFail
0    954
1     14
Name: count, dtype: int64

Test Target
PassOrFail
0    240
1      3
Name: count, dtype: int64


[No Sampling] Logistic Regression
Accuracy : 0.9876543209876543
Precision: 0.0
Recall   : 0.0
F1       : 

,Experiment,Model,Accuracy,Precision,Recall,F1,PR_AUC
0,SMOTE,XGBoost,0.9835,0.0000,0.0000,0.0000,0.5889
1,ADASYN,XGBoost,0.9835,0.0000,0.0000,0.0000,0.5556
2,No Sampling,Random Forest,0.9835,0.0000,0.0000,0.0000,0.4444
3,Class Weight,XGBoost,0.9835,0.0000,0.0000,0.0000,0.4429
4,No Sampling,XGBoost,0.9835,0.0000,0.0000,0.0000,0.4250
5,SMOTE + Tomek,XGBoost,0.9835,0.0000,0.0000,0.0000,0.4206
6,Class Weight,Random Forest,0.9835,0.0000,0.0000,0.0000,0.3889
7,SMOTE + Tomek,Random Forest,0.9835,0.0000,0.0000,0.0000,0.3194
8,SMOTE,Random Forest,0.9835,0.0000,0.0000,0.0000,0.3028
9,ADASYN,Random Forest,0.9835,0.0000,0.0000,0.0000,0.2576



F1 기준


,Experiment,Model,Accuracy,Precision,Recall,F1,PR_AUC
0,SMOTE,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.127496
1,ADASYN,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.127496
2,Class Weight,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.127473
3,SMOTE + Tomek,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.130151
4,SMOTE,XGBoost,0.983539,0.000000,0.000000,0.000000,0.588889
5,ADASYN,XGBoost,0.983539,0.000000,0.000000,0.000000,0.555556
6,No Sampling,Random Forest,0.983539,0.000000,0.000000,0.000000,0.444444
7,Class Weight,Random Forest,0.983539,0.000000,0.000000,0.000000,0.388889
8,SMOTE + Tomek,XGBoost,0.983539,0.000000,0.000000,0.000000,0.420635
9,No Sampling,XGBoost,0.983539,0.000000,0.000000,0.000000,0.425000



Recall 기준


,Experiment,Model,Accuracy,Precision,Recall,F1,PR_AUC
0,SMOTE,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.127496
1,ADASYN,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.127496
2,Class Weight,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.127473
3,SMOTE + Tomek,Logistic Regression,0.958848,0.111111,0.333333,0.166667,0.130151
4,SMOTE,XGBoost,0.983539,0.000000,0.000000,0.000000,0.588889
5,ADASYN,XGBoost,0.983539,0.000000,0.000000,0.000000,0.555556
6,No Sampling,Random Forest,0.983539,0.000000,0.000000,0.000000,0.444444
7,Class Weight,Random Forest,0.983539,0.000000,0.000000,0.000000,0.388889
8,SMOTE + Tomek,XGBoost,0.983539,0.000000,0.000000,0.000000,0.420635
9,No Sampling,XGBoost,0.983539,0.000000,0.000000,0.000000,0.425000




PR-AUC 기준 Best
Experiment : SMOTE
Model      : XGBoost
Precision  : 0.0
Recall     : 0.0
F1         : 0.0
PR-AUC     : 0.5889


In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score
)

# ============================================================
# SMOTE + XGBoost 재학습
# ============================================================

xgb_smote = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_smote.fit(
    X_train_smote,
    y_train_smote
)


# ============================================================
# 예측 확률
# ============================================================

y_proba = xgb_smote.predict_proba(X_test)[:, 1]

print("PR-AUC :", average_precision_score(y_test, y_proba))


# ============================================================
# Threshold별 성능
# ============================================================

threshold_results = []

for threshold in np.arange(0.05, 0.51, 0.05):

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    threshold_results.append({

        "Threshold": threshold,

        "Precision": precision,
        "Recall": recall,
        "F1": f1,

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.style.format({
        "Threshold": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)

PR-AUC : 0.5888888888888889


,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,0.05,0.5000,0.3333,0.4000,1,1,2,239
1,0.10,0.5000,0.3333,0.4000,1,1,2,239
2,0.15,0.0000,0.0000,0.0000,0,1,3,239
3,0.20,0.0000,0.0000,0.0000,0,1,3,239
4,0.25,0.0000,0.0000,0.0000,0,1,3,239
5,0.30,0.0000,0.0000,0.0000,0,1,3,239
6,0.35,0.0000,0.0000,0.0000,0,1,3,239
7,0.40,0.0000,0.0000,0.0000,0,1,3,239
8,0.45,0.0000,0.0000,0.0000,0,1,3,239
9,0.50,0.0000,0.0000,0.0000,0,1,3,239


In [6]:
# ============================================================
# 실제 불량 샘플에 모델이 준 확률 확인
# ============================================================

check = pd.DataFrame({
    "Actual": y_test.values,
    "Probability": y_proba
})

check = check.sort_values(
    "Probability",
    ascending=False
)

display(check.head(30))

,Actual,Probability
233,0,0.992218
127,1,0.134327
98,1,0.023046
13,0,0.022107
157,1,0.020989
116,0,0.017558
111,0,0.017247
115,0,0.016918
190,0,0.016918
200,0,0.016428


In [7]:
display(
    check[
        check["Actual"] == 1
    ].sort_values(
        "Probability",
        ascending=False
    )
)

,Actual,Probability
127,1,0.134327
98,1,0.023046
157,1,0.020989


In [8]:
# ============================================================
# CN7
# Repeated Stratified K-Fold
#
# 비교:
# 1. No Sampling
# 2. Class Weight
# 3. SMOTE
# 4. ADASYN
# 5. SMOTE + Tomek
#
# 모델:
# Logistic / RF / XGBoost
#
# 핵심:
# Sampling은 반드시 각 Fold의 Train에만 적용
# ============================================================


import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek

from xgboost import XGBClassifier


RANDOM_STATE = 42


# ============================================================
# 1. CV 설정
# ============================================================

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=RANDOM_STATE
)


cv_results = []


# ============================================================
# 2. Sampling 함수
# ============================================================

def apply_sampling(
    method,
    X_train,
    y_train
):

    minority_count = y_train.value_counts().min()

    k = max(
        1,
        min(
            5,
            minority_count - 1
        )
    )


    # --------------------------------
    # Sampling 없음
    # --------------------------------

    if method == "No Sampling":

        return (
            X_train.copy(),
            y_train.copy()
        )


    # --------------------------------
    # Class Weight
    # Sampling 자체는 없음
    # --------------------------------

    if method == "Class Weight":

        return (
            X_train.copy(),
            y_train.copy()
        )


    # --------------------------------
    # SMOTE
    # --------------------------------

    if method == "SMOTE":

        sampler = SMOTE(
            random_state=RANDOM_STATE,
            k_neighbors=k
        )


    # --------------------------------
    # ADASYN
    # --------------------------------

    elif method == "ADASYN":

        sampler = ADASYN(
            random_state=RANDOM_STATE,
            n_neighbors=k
        )


    # --------------------------------
    # SMOTE + Tomek
    # --------------------------------

    elif method == "SMOTE + Tomek":

        sampler = SMOTETomek(

            random_state=RANDOM_STATE,

            smote=SMOTE(
                random_state=RANDOM_STATE,
                k_neighbors=k
            )
        )


    return sampler.fit_resample(
        X_train,
        y_train
    )


# ============================================================
# 3. 모델 생성
# ============================================================

def get_models(
    method,
    y_train
):

    # --------------------------------
    # Class Weight
    # --------------------------------

    if method == "Class Weight":

        class_weight = "balanced"

    else:

        class_weight = None


    # --------------------------------
    # XGB scale_pos_weight
    # --------------------------------

    if method == "Class Weight":

        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()

        scale_pos_weight = (
            neg / pos
        )

    else:

        scale_pos_weight = 1


    models = {

        "Logistic Regression":

            LogisticRegression(
                max_iter=3000,
                class_weight=class_weight,
                random_state=RANDOM_STATE
            ),


        "Random Forest":

            RandomForestClassifier(
                n_estimators=500,
                class_weight=class_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),


        "XGBoost":

            XGBClassifier(

                n_estimators=300,

                max_depth=4,

                learning_rate=0.05,

                subsample=0.8,

                colsample_bytree=0.8,

                scale_pos_weight=scale_pos_weight,

                objective="binary:logistic",

                eval_metric="logloss",

                random_state=RANDOM_STATE,

                n_jobs=-1
            )
    }


    return models


# ============================================================
# 4. 실험
# ============================================================

sampling_methods = [

    "No Sampling",
    "Class Weight",
    "SMOTE",
    "ADASYN",
    "SMOTE + Tomek"

]


for method in sampling_methods:

    print("\n")
    print("=" * 70)
    print(method)
    print("=" * 70)


    fold_num = 1


    for train_idx, val_idx in cv.split(X, y):


        # --------------------------------
        # Fold 분리
        # --------------------------------

        X_tr = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()

        y_tr = y.iloc[train_idx].copy()
        y_val = y.iloc[val_idx].copy()


        # --------------------------------
        # Sampling
        # Train에만 적용
        # --------------------------------

        try:

            X_res, y_res = apply_sampling(
                method,
                X_tr,
                y_tr
            )

        except ValueError as e:

            print(
                f"Fold {fold_num} "
                f"{method} 실패:",
                e
            )

            fold_num += 1
            continue


        # --------------------------------
        # 모델
        # --------------------------------

        models = get_models(
            method,
            y_tr
        )


        for model_name, model in models.items():


            # 학습
            model.fit(
                X_res,
                y_res
            )


            # 확률
            proba = (
                model
                .predict_proba(X_val)[:, 1]
            )


            # 기본 threshold
            pred = (
                proba >= 0.5
            ).astype(int)


            # 지표
            precision = precision_score(
                y_val,
                pred,
                zero_division=0
            )

            recall = recall_score(
                y_val,
                pred,
                zero_division=0
            )

            f1 = f1_score(
                y_val,
                pred,
                zero_division=0
            )

            pr_auc = average_precision_score(
                y_val,
                proba
            )


            cv_results.append({

                "Sampling": method,

                "Model": model_name,

                "Fold": fold_num,

                "Precision": precision,

                "Recall": recall,

                "F1": f1,

                "PR_AUC": pr_auc

            })


        fold_num += 1


# ============================================================
# 5. Fold 결과
# ============================================================

cv_results_df = pd.DataFrame(
    cv_results
)


display(
    cv_results_df.head()
)


# ============================================================
# 6. 평균 + 표준편차
# ============================================================

summary = (

    cv_results_df

    .groupby(
        [
            "Sampling",
            "Model"
        ]
    )

    .agg({

        "Precision": [
            "mean",
            "std"
        ],

        "Recall": [
            "mean",
            "std"
        ],

        "F1": [
            "mean",
            "std"
        ],

        "PR_AUC": [
            "mean",
            "std"
        ]

    })

)


summary.columns = [

    "Precision_Mean",
    "Precision_STD",

    "Recall_Mean",
    "Recall_STD",

    "F1_Mean",
    "F1_STD",

    "PR_AUC_Mean",
    "PR_AUC_STD"

]


summary = (

    summary

    .reset_index()

    .sort_values(
        "PR_AUC_Mean",
        ascending=False
    )

)


# ============================================================
# 7. 결과 출력
# ============================================================

display(

    summary.style.format({

        "Precision_Mean": "{:.4f}",
        "Precision_STD": "{:.4f}",

        "Recall_Mean": "{:.4f}",
        "Recall_STD": "{:.4f}",

        "F1_Mean": "{:.4f}",
        "F1_STD": "{:.4f}",

        "PR_AUC_Mean": "{:.4f}",
        "PR_AUC_STD": "{:.4f}"

    })

)



No Sampling


Class Weight


SMOTE


ADASYN


SMOTE + Tomek


,Sampling,Model,Fold,Precision,Recall,F1,PR_AUC
0,No Sampling,Logistic Regression,1,0.500000,0.500000,0.500000,0.568750
1,No Sampling,Random Forest,1,0.333333,0.500000,0.400000,0.581155
2,No Sampling,XGBoost,1,0.333333,0.500000,0.400000,0.399306
3,No Sampling,Logistic Regression,2,1.000000,0.333333,0.500000,0.505128
4,No Sampling,Random Forest,2,0.250000,0.333333,0.285714,0.465812


,Sampling,Model,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,PR_AUC_Mean,PR_AUC_STD
6,No Sampling,Logistic Regression,0.6700,0.4170,0.3633,0.2578,0.4483,0.2854,0.5041,0.2310
8,No Sampling,XGBoost,0.3648,0.2999,0.2933,0.2193,0.3048,0.2164,0.4810,0.1565
12,SMOTE + Tomek,Logistic Regression,0.1754,0.0980,0.5567,0.2708,0.2618,0.1339,0.4800,0.2357
3,Class Weight,Logistic Regression,0.1679,0.0928,0.5900,0.2534,0.2557,0.1244,0.4759,0.2362
9,SMOTE,Logistic Regression,0.1767,0.0986,0.5567,0.2708,0.2634,0.1349,0.4746,0.2375
1,ADASYN,Random Forest,0.3795,0.2926,0.3400,0.2268,0.3333,0.2128,0.4626,0.1847
7,No Sampling,Random Forest,0.3869,0.2939,0.3400,0.2380,0.3360,0.2186,0.4625,0.2243
5,Class Weight,XGBoost,0.3623,0.2792,0.3633,0.2475,0.3326,0.2068,0.4532,0.1519
0,ADASYN,Logistic Regression,0.1792,0.0862,0.5933,0.2469,0.2712,0.1201,0.4450,0.2319
11,SMOTE,XGBoost,0.3890,0.2920,0.3833,0.2774,0.3554,0.2291,0.4422,0.1889
